# MLP

A simple multi-layer perceptron (MLP) built and trained from scratch, with animated visualisations of the training process and inference.

A single perceptron has a weight vector $w$, a bias $b$, and a step-function activation. It learns by nudging its weights directly toward the correct output — a simple rule that works for linearly separable problems but cannot be extended to multiple layers.

There are several key differences in the multi-layer perceptron:

1. **Activation function** — The single perceptron learning rule works by directly computing the error and nudging the weights. This doesn't generalise to a chain of layers. MLPs need to propagate the error backwards (backpropagation) using the chain rule, which requires a differentiable activation function — sigmoid, tanh, or ReLU rather than a step function.

2. **Weights become matrices** — A single perceptron has a weight vector of size $n$. A layer of $k$ neurons each receiving $n$ inputs has a weight matrix of shape $(n, k)$. Forward passes become matrix multiplications: $Z = XW + b$.

3. **Backpropagation** — Starting from the output loss, each layer computes $\frac{\partial L}{\partial W}$ by chaining gradients backwards through the network using its own activation derivative and the gradient flowing in from the layer ahead.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from sklearn.datasets import load_iris

from activation_functions import (
    sigmoid, ReLU, tanh,
    sigmoid_derivative, relu_derivative, tanh_derivative,
)
from loss_functions import mse, mse_d

## Layer

An MLP consists of one or more hidden layers chaining input to output. Each `Layer` holds a weight matrix of shape $(n_{inputs}, n_{nodes})$, a bias vector of shape $(n_{nodes},)$, and a choice of activation function.

**Forward pass:** $Z = XW + b$, then $A = \text{activation}(Z)$. The output of each layer becomes the input to the next.

**Backward pass (backpropagation):** Starting from the loss gradient $\frac{\partial L}{\partial A}$, each layer applies the chain rule to compute $\frac{\partial L}{\partial W}$, $\frac{\partial L}{\partial b}$, and $\frac{\partial L}{\partial X}$ (passed to the previous layer), then updates its own weights and bias in place.

In [ ]:
class Layer:

    def __init__(self,
        n_inputs: int = 4,
        n_nodes: int = 8,
        learning_rate: float = 0.01,
        activation_fn: callable = sigmoid,
        activation_derivative: callable = sigmoid_derivative,
    ):
        self.n_inputs = n_inputs
        self.n_nodes = n_nodes
        self.learning_rate = learning_rate
        self.activation_fn = activation_fn
        self.activation_derivative = activation_derivative
        self.bias: np.ndarray = np.zeros(n_nodes)
        self.initialize_weights()

    def initialize_weights(self):
        self.weight_matrix: np.ndarray = np.random.uniform(-1, 1, (self.n_inputs, self.n_nodes))

    def forward_pass(self, X: np.ndarray) -> np.ndarray:
        self.input = X
        self.z = X @ self.weight_matrix + self.bias
        return self.activation_fn(self.z)

    def backpropagation(self, d_out: np.ndarray) -> np.ndarray:
        d_z = d_out * self.activation_derivative(self.z)
        d_W = np.outer(self.input, d_z)
        d_input = d_z @ self.weight_matrix.T
        self.weight_matrix -= self.learning_rate * d_W
        self.bias -= self.learning_rate * d_z
        return d_input

The `MLP` class is a thin container over a list of `Layer` objects. Training is a forward pass to get a prediction, a loss computation, and a backward pass that walks the layers in reverse — each layer updating its own weights and passing the gradient back to the one before it.

In [ ]:
class MLP:

    def __init__(self,
        layers: list[Layer],
        loss_fn: callable = mse,
        loss_fn_d: callable = mse_d,
    ):
        self.layers = layers
        self.loss_fn = loss_fn
        self.loss_fn_d = loss_fn_d

    def forward(self, X: np.ndarray) -> np.ndarray:
        out = X
        for layer in self.layers:
            out = layer.forward_pass(out)
        return out

    def backward(self, d_loss: np.ndarray):
        grad = d_loss
        for layer in reversed(self.layers):
            grad = layer.backpropagation(grad)

    def train(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        y_pred = self.forward(X)
        d_loss = self.loss_fn_d(y_pred, y)
        self.backward(d_loss)
        return self.loss_fn(y_pred, y)

## Dataset

Iris versicolor vs virginica — a binary classification problem with 4 features and overlapping class distributions, making it a more realistic test than a trivially separable dataset. Features are z-score normalised and split 80/20 train/test.

In [ ]:
# Versicolor (0) vs Virginica (1) — overlapping classes, not trivially separable
iris = load_iris()
mask = iris.target != 0               # drop setosa
X_all = iris.data[mask].astype(float)
y_all = (iris.target[mask] == 2).astype(int)

# Z-score normalise each feature
X_all = (X_all - X_all.mean(axis=0)) / X_all.std(axis=0)

# Shuffle and split 80/20
np.random.seed(42)
idx = np.random.permutation(len(X_all))
split = int(len(X_all) * 0.8)
X_train, X_test = X_all[idx[:split]], X_all[idx[split:]]
y_train, y_test = y_all[idx[:split]], y_all[idx[split:]]

N_SAMPLES = len(X_train)
print(f"Train: {N_SAMPLES}  Test: {len(X_test)}  Features: {X_train.shape[1]}")

## Training

Architecture: `4 → 8 → 1`. After each epoch we snapshot the weights and a fixed sample's activations for use in the visualisation below.

In [ ]:
np.random.seed(42)
mlp = MLP([
    Layer(n_inputs=4, n_nodes=8, learning_rate=0.05),
    Layer(n_inputs=8, n_nodes=1, learning_rate=0.05),
])

n_epochs = 80
epoch_data = []
loss_history = []
acc_history = []
sample_x = X_train[0]

for epoch in range(n_epochs):
    total_loss = 0
    for x, y in zip(X_train, y_train):
        total_loss += mlp.train(x, y).sum()
    loss_history.append(total_loss / N_SAMPLES)

    # Training accuracy (threshold output at 0.5)
    preds = [(mlp.forward(x)[0] >= 0.5) for x in X_train]
    acc_history.append(sum(p == y for p, y in zip(preds, y_train)) / N_SAMPLES)

    # Capture activations for a fixed sample
    activations = [sample_x.copy()]
    xi = sample_x.copy()
    for layer in mlp.layers:
        xi = layer.forward_pass(xi)
        activations.append(xi.copy())

    epoch_data.append({
        'activations': activations,
        'weights': [l.weight_matrix.copy() for l in mlp.layers],
    })

# Test accuracy
test_preds = [(mlp.forward(x)[0] >= 0.5) for x in X_test]
test_acc = sum(p == y for p, y in zip(test_preds, y_test)) / len(y_test)
print(f"Final train loss: {loss_history[-1]:.4f}  |  Test accuracy: {test_acc:.0%}")

In [ ]:
LAYER_SIZES  = [4, 8, 1]
LAYER_LABELS = ['Input\n(4 features)', 'Hidden\n(8 nodes)', 'Output']
H_GAP, V_GAP, R = 6.0, 1.4, 0.35

def get_node_positions(sizes):
    max_n = max(sizes)
    return [
        [(li * H_GAP, (max_n - n) / 2 * V_GAP + ni * V_GAP)
         for ni in range(n)]
        for li, n in enumerate(sizes)
    ]

NODE_POS = get_node_positions(LAYER_SIZES)
Y_MIN = min(p[1] for layer in NODE_POS for p in layer)
Y_MAX = max(p[1] for layer in NODE_POS for p in layer)
X_MIN = -2.5   # room for input feature labels
X_MAX = H_GAP * (len(LAYER_SIZES) - 1) + 1.0

fig = plt.figure(figsize=(20, 12))
gs  = fig.add_gridspec(2, 2, height_ratios=[2.5, 1], hspace=0.35, wspace=0.3)
ax_net  = fig.add_subplot(gs[0, :])
ax_acc  = fig.add_subplot(gs[1, 0])
ax_loss = fig.add_subplot(gs[1, 1])

# Accuracy axis
ax_acc.set_xlim(0, n_epochs + 1)
ax_acc.set_ylim(0, 1.05)
ax_acc.set_xlabel('Epoch')
ax_acc.set_ylabel('Accuracy')
ax_acc.set_title('Training Accuracy')
ax_acc.axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
acc_line, = ax_acc.plot([], [], color='#27ae60', marker='o', markersize=3, linewidth=1.5)

# Loss axis
ax_loss.set_xlim(0, n_epochs + 1)
ax_loss.set_ylim(0, max(loss_history) * 1.15)
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel('MSE Loss')
ax_loss.set_title('Training Loss')
ax_loss.axhline(0, color='grey', linewidth=0.5)
loss_line, = ax_loss.plot([], [], color='#e67e22', marker='o', markersize=3, linewidth=1.5)

FEATURE_NAMES = ['sepal len', 'sepal wid', 'petal len', 'petal wid']

def draw_frame(i):
    ax_net.clear()
    ax_net.axis('off')
    ax_net.set_xlim(X_MIN, X_MAX)
    ax_net.set_ylim(Y_MIN - 1.2, Y_MAX + 0.5)

    acts   = epoch_data[i]['activations']
    W_list = epoch_data[i]['weights']
    max_w  = max(np.max(np.abs(W)) for W in W_list) + 1e-8

    # Edges — blue = positive, red = negative; thickness & opacity ∝ magnitude
    for li, W in enumerate(W_list):
        for ni, (x0, y0) in enumerate(NODE_POS[li]):
            for nj, (x1, y1) in enumerate(NODE_POS[li + 1]):
                w = W[ni, nj]
                ax_net.plot([x0, x1], [y0, y1],
                            color='#2471a3' if w >= 0 else '#c0392b',
                            alpha=float(np.clip(abs(w) / max_w, 0.05, 0.85)),
                            linewidth=float(abs(w) / max_w * 3.0),
                            zorder=1)

    # Nodes — YlOrRd colormap, activation value labelled inside
    for li, layer_pos in enumerate(NODE_POS):
        for ni, (x, y) in enumerate(layer_pos):
            val  = float(acts[li][ni])
            norm = float(np.clip((val + 2) / 4 if li == 0 else val, 0, 1))
            circle = plt.Circle((x, y), R,
                                 facecolor=plt.cm.YlOrRd(norm),
                                 edgecolor='#2c3e50', linewidth=1.5, zorder=2)
            ax_net.add_patch(circle)
            ax_net.text(x, y, f'{val:.2f}', ha='center', va='center',
                        fontsize=8, fontweight='bold', zorder=3)
            if li == 0:
                ax_net.text(x - R - 0.15, y, FEATURE_NAMES[ni],
                            ha='right', va='center', fontsize=9, color='#555')

    for li, label in enumerate(LAYER_LABELS):
        ax_net.text(li * H_GAP, Y_MIN - R - 0.65, label,
                    ha='center', fontsize=10, color='#444')

    ax_net.set_title(
        f'Epoch {i + 1}   |   Loss: {loss_history[i]:.4f}   |   Acc: {acc_history[i]:.0%}',
        fontsize=12, pad=10)

    acc_line.set_data(range(1, i + 2), acc_history[:i + 1])
    loss_line.set_data(range(1, i + 2), loss_history[:i + 1])
    return []

anim = animation.FuncAnimation(fig, draw_frame, frames=n_epochs, interval=250, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# Run inference on all test samples and capture per-layer activations
test_acts = []
for x in X_test:
    acts = [x.copy()]
    xi   = x.copy()
    for layer in mlp.layers:
        xi = layer.forward_pass(xi)
        acts.append(xi.copy())
    test_acts.append(acts)

CLASS_NAMES       = ['Versicolor', 'Virginica']
FRAMES_PER_SAMPLE = 3   # 0=input  1=hidden  2=output+result
FIRE_THRESHOLD    = 0.5
n_test            = len(X_test)
total_frames      = n_test * FRAMES_PER_SAMPLE
max_w_inf         = max(np.max(np.abs(l.weight_matrix)) for l in mlp.layers) + 1e-8

STAGE_LABELS = ['Input received', 'Hidden layer activating', 'Output computed']

fig_inf, ax_inf = plt.subplots(figsize=(20, 8))

def draw_inference_frame(frame):
    ax_inf.clear()
    ax_inf.axis('off')
    ax_inf.set_xlim(X_MIN, X_MAX)
    ax_inf.set_ylim(Y_MIN - 1.5, Y_MAX + 1.2)

    si    = frame // FRAMES_PER_SAMPLE
    stage = frame %  FRAMES_PER_SAMPLE

    acts   = test_acts[si]
    W_list = [l.weight_matrix for l in mlp.layers]

    lit_layers = {0} | ({1} if stage >= 1 else set()) | ({2} if stage >= 2 else set())
    lit_edges  = ({0} if stage >= 1 else set()) | ({1} if stage >= 2 else set())

    # Which hidden nodes are firing (used to highlight their outgoing edges at stage 2)
    hidden_acts  = acts[1]
    firing_hidden = {ni for ni, v in enumerate(hidden_acts) if float(v) > FIRE_THRESHOLD}

    # Edges
    for li, W in enumerate(W_list):
        for ni, (x0, y0) in enumerate(NODE_POS[li]):
            for nj, (x1, y1) in enumerate(NODE_POS[li + 1]):
                w = W[ni, nj]
                if li in lit_edges:
                    # At stage 2: edges FROM firing hidden nodes are brighter & thicker
                    is_fired_source = (li == 1 and ni in firing_hidden)
                    base_alpha = float(np.clip(abs(w) / max_w_inf, 0.05, 0.85))
                    color = '#2471a3' if w >= 0 else '#c0392b'
                    if is_fired_source:
                        alpha = min(base_alpha * 1.8, 1.0)
                        lw    = float(abs(w) / max_w_inf * 5.0)
                    else:
                        alpha = base_alpha
                        lw    = float(abs(w) / max_w_inf * 3.0)
                else:
                    color, alpha, lw = '#cccccc', 0.12, 0.5
                ax_inf.plot([x0, x1], [y0, y1],
                            color=color, alpha=alpha, linewidth=lw, zorder=1)

    # Nodes
    for li, layer_pos in enumerate(NODE_POS):
        for ni, (x, y) in enumerate(layer_pos):
            if li in lit_layers:
                val  = float(acts[li][ni])
                norm = float(np.clip((val + 2) / 4 if li == 0 else val, 0, 1))
                fc   = plt.cm.YlOrRd(norm)
                txt  = f'{val:.2f}'

                # Firing indicator: bright border + thicker ring for hidden/output nodes
                firing = (li > 0) and (val > FIRE_THRESHOLD)
                ec  = "#12f312" if firing else '#2c3e50'
                elw = 3.5       if firing else 1.5
            else:
                fc, txt, ec, elw = '#eeeeee', '', '#2c3e50', 1.5

            circle = plt.Circle((x, y), R, facecolor=fc,
                                 edgecolor=ec, linewidth=elw, zorder=2)
            ax_inf.add_patch(circle)
            if txt:
                ax_inf.text(x, y, txt, ha='center', va='center',
                            fontsize=8, fontweight='bold', zorder=3)
            if li == 0:
                ax_inf.text(x - R - 0.15, y, FEATURE_NAMES[ni],
                            ha='right', va='center', fontsize=9, color='#555')

    # Layer labels
    for li, label in enumerate(LAYER_LABELS):
        ax_inf.text(li * H_GAP, Y_MIN - R - 0.65, label,
                    ha='center', fontsize=10, color='#444')

    # Legend
    ax_inf.text(X_MAX - 0.2, Y_MAX + 0.3,
                '█  firing (> 0.5)', color='#f39c12', fontsize=9, ha='right')

    # Result — shown at stage 2
    if stage == 2:
        out_val = float(acts[-1][0])
        pred    = int(out_val >= 0.5)
        true    = int(y_test[si])
        correct = pred == true
        ax_inf.text(
            (X_MIN + X_MAX) / 2, Y_MAX + 0.85,
            f"Predicted: {CLASS_NAMES[pred]}  (confidence {out_val:.2f})   "
            f"·   True: {CLASS_NAMES[true]}   {'✓' if correct else '✗'}",
            ha='center', fontsize=12, fontweight='bold',
            color='#27ae60' if correct else '#e74c3c')

    ax_inf.set_title(
        f'Test sample {si + 1} / {n_test}   ·   {STAGE_LABELS[stage]}',
        fontsize=13, pad=12)

anim_inf = animation.FuncAnimation(
    fig_inf, draw_inference_frame,
    frames=total_frames, interval=700, blit=False
)
plt.close(fig_inf)
HTML(anim_inf.to_jshtml())